# Notebook 5 / 8 — Non-Classical Logics in Language Tasks

> **Series.** This is the fifth of eight notebooks exploring how non-classical logics help agents communicate.
> 1. *Basics* — eight foundational logics, one short scenario each
> 2. *Advanced* — fourteen rarer logics with richer expressive power
> 3. *Synthesis* — cross-logic benchmarks and research-style conclusions
> 4. *Applications* — ten general-purpose domains where each logic earns its keep
> 5. **Language** *(this notebook)* — non-classical logics applied to natural-language tasks
> 6. *Workflow* — an end-to-end pipeline composing the best of the above
> 7. *LangGraph* — the same pipeline rebuilt as a LangGraph state machine
> 8. *Experimental composition* — probing what happens when two non-classical logics overlap on the same linguistic phenomenon

## What this notebook is

Natural language is the place where the limits of classical logic show up most often: speakers hedge, contradict each other, retract, refer to things that don't exist, mean different things by the same word, and rely on common ground that no formal model captures. This notebook walks through ten **language tasks** where a small dose of non-classical logic produces a measurably better outcome than a naive boolean / probabilistic model.

Each section follows the same shape:
1. **Task** — the language phenomenon
2. **The bug** — what classical NLP / boolean reasoning gets wrong
3. **The fix** — minimal evaluator using the appropriate logic
4. **What it repairs** — concrete linguistic behaviour

## The logics applied here — definitions and references

### 1. Fuzzy logic (vague predicates)
**Definition.** Truth values form the real interval `[0,1]`. Connectives are *t-norms* (∧) and *t-conorms* (∨). Linguistic *hedges* are unary operators on the value: Zadeh's `very` is `μ²`, `somewhat` is `√μ`. Designed in part to model the semantics of vague natural-language predicates such as *tall*, *warm*, *young*.
**Reference.** Zadeh, L. A. (1965). *Fuzzy sets*. Information and Control 8(3), 338–353. Zadeh, L. A. (1972). *A fuzzy-set-theoretic interpretation of linguistic hedges*. Journal of Cybernetics 2(3), 4–34.

### 2. Paraconsistent logic — LP (contradictory dialogue)
**Definition.** Three-valued logic over `{F}, {T}, {T,F}`; a formula is *designated* iff `T` is in its value. Crucially, `p ∧ ¬p` is satisfiable, so the rule `p, ¬p ⊢ q` (ex falso quodlibet) fails. This makes LP a natural backbone for *discourse representation* under speaker disagreement.
**Reference.** Priest, G. (1979). *The logic of paradox*. Journal of Philosophical Logic 8(1), 219–241. Priest, G. (2006). *In Contradiction* (2nd ed.). OUP.

### 3. Free logic (presupposition failure)
**Definition.** A first-order logic in which singular terms may fail to denote. The existence predicate `E!t` is first-class. *Negative* free logic treats any atomic formula with a non-denoting term as false; *neutral* (or "presuppositional") free logic returns a third value `*`. Strawson's analysis of definite descriptions is the classic motivation.
**Reference.** Strawson, P. F. (1950). *On referring*. Mind 59(235), 320–344. Lambert, K. (1967). *Free logic and the concept of existence*. Notre Dame Journal of Formal Logic 8, 133–144. Survey: Bencivenga, E. (1986). *Free logics*. Handbook of Philosophical Logic III, 373–426.

### 4. Default logic (conversational implicature)
**Definition.** A non-monotonic logic. A *default* `α : β / γ` reads "if `α` is known and `β` is consistent with the current beliefs, conclude `γ`". An *extension* of a knowledge base is a fixed point under such rules; adding new facts can shrink an extension, modelling retractable conversational implicatures.
**Reference.** Reiter, R. (1980). *A logic for default reasoning*. Artificial Intelligence 13(1–2), 81–132. Linguistic application: Levinson, S. C. (2000). *Presumptive Meanings*. MIT.

### 5. Epistemic logic (nested belief reports)
**Definition.** A multi-agent modal logic. `K_i φ` reads "agent `i` knows that `φ`"; iterated forms `K_a K_b φ` express nested attitudes ("Alice knows that Bob knows"). Interpreted over multi-relational Kripke frames, one accessibility relation per agent.
**Reference.** Hintikka, J. (1962). *Knowledge and Belief*. Cornell University Press. Fagin, Halpern, Moses & Vardi (1995). *Reasoning about Knowledge*. MIT Press.

### 6. Linear Temporal Logic — LTL (tense and aspect)
**Definition.** A propositional temporal logic interpreted over linear traces. Operators `X` (next), `F` (eventually), `G` (always), `U` (until) align directly with English tense and aspect markers. Used in linguistic semantics under the names *Priorean tense logic* and its successors.
**Reference.** Pnueli, A. (1977). *The temporal logic of programs*. FOCS '77, 46–57. Linguistic precursor: Prior, A. N. (1967). *Past, Present and Future*. Oxford.

### 7. Possibilistic logic (translation under hedging)
**Definition.** A weighted classical KB `{(φ_i, α_i)}` with `α_i ∈ [0,1]` interpreted as necessity. Inference rules propagate weights by taking the minimum along resolution chains. Linguistic hedges (*allegedly*, *reportedly*) act as multiplicative discounts on the weight.
**Reference.** Dubois, D., Lang, J. & Prade, H. (1994). *Possibilistic logic*. Handbook of Logic in AI and Logic Programming 3, 439–513.

### 8. Subjective logic (annotator disagreement / moderation)
**Definition.** A calculus of *opinions* `ω = (b, d, u, a)` over a binary frame, with `b + d + u = 1`. Cumulative fusion of independent opinions: `b' = (b₁u₂ + b₂u₁) / (u₁ + u₂ − u₁u₂)`. Designed precisely for combining noisy human or machine judgements while preserving residual uncertainty.
**Reference.** Jøsang, A. (2001). *A logic for uncertain probabilities*. International Journal of Uncertainty, Fuzziness and Knowledge-Based Systems 9(3), 279–311. Book: Jøsang, *Subjective Logic* (Springer 2016).

### 9. Dempster–Shafer theory (word-sense fusion)
**Definition.** Belief mass `m: 2^Ω → [0,1]` is distributed over *subsets* of a frame of discernment `Ω`. Dempster's rule combines two independent mass functions and renormalises by the conflict mass `K`. A natural fit when contextual cues identify *families* of senses rather than a single one.
**Reference.** Dempster, A. P. (1967). *Upper and lower probabilities induced by a multivalued mapping*. Annals of Mathematical Statistics 38, 325–339. Shafer, G. (1976). *A Mathematical Theory of Evidence*. Princeton.

### 10. Intuitionistic logic (constructive QA / RAG)
**Definition.** A constructive logic in which `φ` holds at an information state `s` only if there is a constructive witness; `φ ∨ ¬φ` is *not* a tautology. Kripke's semantics interprets information states as nodes in a partial order; truth is monotonic — once established, it persists. Mapping retrieved passages to information states gives a principled anti-hallucination mechanism.
**Reference.** Heyting, A. (1930). *Die formalen Regeln der intuitionistischen Logik*. Sitzungsberichte der Preußischen Akademie. Kripke, S. A. (1965). *Semantical analysis of intuitionistic logic I*. Formal Systems and Recursive Functions, 92–130.

## Quick-glance table

| § | Language task | Logic |
|---|---|---|
| 1 | Vague predicates | Fuzzy / Łukasiewicz |
| 2 | Contradictory dialogue | Paraconsistent LP |
| 3 | Presupposition failure | Free logic |
| 4 | Implicature & defaults | Default logic |
| 5 | Nested belief reports | Epistemic logic |
| 6 | Tense & aspect | LTL |
| 7 | Hedged translation | Possibilistic |
| 8 | Annotator disagreement | Subjective logic |
| 9 | Word-sense fusion | Dempster–Shafer |
| 10 | Grounded QA / RAG | Intuitionistic |

Each section is self-contained. No external NLP libraries are used — the focus is on the *logic*, not on a particular embedding model.

In [ ]:
from dataclasses import dataclass, field
from itertools import product, combinations
from typing import Dict, List, Set, Tuple, Optional, FrozenSet
from functools import reduce
from collections import Counter
import math, random

random.seed(21)

def task(name, logic):
    bar = "━" * 64
    print(f"\n{bar}\n  {name}\n  → {logic}\n{bar}")

## 1. Vague predicates — "tall", "warm", "old"

**Task.** Decide whether sentences like *"Alice is tall"* are true. The predicate `tall` has no sharp threshold — there is no height at which `tall(h) = False` jumps to `tall(h+1ε) = True`.

**The bug (classical).** A boolean threshold (e.g. `height > 180`) creates the *sorites paradox*: removing a millimetre at a time from a tall person never produces a non-tall person. Worse, hedges like *very tall*, *fairly tall*, *not particularly tall* collapse to the same `True/False`.

**The fix.** Fuzzy / Łukasiewicz truth values, with hedges as truth-modifying functions.

In [ ]:
task("vague predicates", "fuzzy logic")

def tall(height_cm):
    # Smooth membership: 0 below 160, 1 above 195, linear in between.
    if height_cm <= 160: return 0.0
    if height_cm >= 195: return 1.0
    return (height_cm - 160) / 35

def very(mu):       return mu ** 2          # Zadeh's intensifier
def somewhat(mu):   return mu ** 0.5
def not_(mu):       return 1 - mu

for h in [158, 170, 178, 185, 194, 200]:
    mu = tall(h)
    print(f"  {h} cm  tall={mu:.2f}  very_tall={very(mu):.2f}  somewhat_tall={somewhat(mu):.2f}  not_tall={not_(mu):.2f}")

print("\nFix: hedges modulate the same underlying value, so 'very tall' implies 'tall'")
print("and the sorites series degrades smoothly instead of flipping at a phantom threshold.")

## 2. Speaker disagreement — discourse with contradictions

**Task.** Build a discourse representation from a conversation where speakers contradict each other. A dialogue agent must record both utterances *as said* without losing the ability to keep talking about other topics.

**The bug (classical).** A boolean discourse model with `says(A, p)` and `says(B, ¬p)` is fine, but the moment the agent tries to *believe* the union of asserted content, it explodes: `p ∧ ¬p ⊢ q` for any `q`, so the agent suddenly "believes" arbitrary nonsense.

**The fix.** LP (paraconsistent). The merged content lives in `{T, F, T∧F}`. A literal flagged `T∧F` is reported back as "sources disagree" rather than triggering downstream nonsense.

In [ ]:
task("contradictory dialogue", "paraconsistent LP")

Tv = frozenset({"T"}); Fv = frozenset({"F"}); Bv = frozenset({"T","F"})

def merge(a, b): return frozenset(a | b)
def lp_or(a, b):
    out = set()
    if "T" in a or "T" in b: out.add("T")
    if "F" in a and "F" in b: out.add("F")
    return frozenset(out)

discourse = {}
utterances = [
    ("Alice", "the_meeting_is_today", Tv),
    ("Bob",   "the_meeting_is_today", Fv),
    ("Alice", "agenda_is_finalized",  Tv),
    ("Carol", "projector_works",      Tv),
]
for spkr, prop, val in utterances:
    discourse[prop] = merge(discourse.get(prop, frozenset()), val)
    print(f"  {spkr:6s} asserts {prop:25s} -> stored as {set(discourse[prop])}")

for prop, val in discourse.items():
    if val == Bv:
        print(f"\n  ⚠ '{prop}' is contested — surface to user, do not silently pick a side.")
    elif val == Tv:
        print(f"  ✓ '{prop}' agreed.")

print("\nFix: the agent can answer 'is the agenda finalized?' (yes) and 'is the meeting today?'")
print("(disputed) without one contradiction poisoning the entire discourse model.")

## 3. Presupposition failure — "the king of France is bald"

**Task.** Evaluate sentences whose subject does not refer. Russell vs Strawson: should *"The king of France is bald"* be False (there is no such king, so the predication fails) or *neither true nor false* (the presupposition that there is a king fails)?

**The bug (classical).** Forcing every term to denote either invents a phantom king (so the sentence is vacuously True for any predicate) or makes the sentence False, which then validates *"The king of France is not bald"* — also nonsense.

**The fix.** Free logic. Predicates over non-denoting terms return a third value `*` that downstream code can route to a presupposition-failure response ("there is no king of France").

In [ ]:
task("presupposition failure", "free logic")

@dataclass
class FreeModel:
    domain: Set[str]
    interp: Dict[str, Set[str]]
    references: Dict[str, Optional[str]]   # definite description -> referent (may be None)

    def E(self, t): return t in self.domain
    def deref(self, desc): return self.references.get(desc)
    def pred(self, P, desc):
        t = self.deref(desc)
        if t is None or not self.E(t): return "*"
        return t in self.interp.get(P, set())

M = FreeModel(
    domain={"emmanuel_macron", "charles_iii"},
    interp={"bald": {"charles_iii"}, "president": {"emmanuel_macron"}},
    references={
        "the_king_of_france": None,            # presupposition failure
        "the_president_of_france": "emmanuel_macron",
        "the_king_of_england":  "charles_iii",
    },
)

tests = [
    ("bald", "the_king_of_france"),
    ("bald", "the_king_of_england"),
    ("bald", "the_president_of_france"),
]
for P, desc in tests:
    val = M.pred(P, desc)
    label = {True:"True", False:"False", "*":"* (presupposition failure)"}[val]
    print(f"  {P}({desc}) = {label}")

print("\nFix: the system answers 'there is no king of France' instead of asserting either")
print("that he is bald or that he is not bald.")

## 4. Conversational implicature & defaults

**Task.** Resolve sentences that rely on a default interpretation overridable by context. *"Tweety is a bird"* implies *"Tweety flies"*; *"Tweety is a penguin"* withdraws that implication.

**The bug (classical).** Monotonic logic: once you derived `flies(tweety)`, no amount of new information can take it back. Standard NLP pipelines patch this with hard-coded exception lists, which scale badly.

**The fix.** Default logic. The default `bird : flies / flies` fires only when `flies` is consistent with the rest of the KB; the additional fact `not_flies` blocks it.

In [ ]:
task("defaults & implicature", "default logic")

@dataclass
class Default:
    pre: str
    justification: str
    conclusion: str

def extension(facts, defaults):
    out = set(facts)
    changed = True
    while changed:
        changed = False
        for d in defaults:
            if d.pre in out and ("not_"+d.justification) not in out and d.conclusion not in out:
                out.add(d.conclusion); changed = True
    return out

defaults = [
    Default("bird",     "flies",   "flies"),
    Default("professor","teaches", "teaches"),    # 'professor' implicates 'teaches'
]

for label, facts in [
    ("'Tweety is a bird.'",                              {"bird"}),
    ("'Tweety is a bird. Tweety is a penguin.'",         {"bird","penguin","not_flies"}),
    ("'Dr Smith is a professor.'",                       {"professor"}),
    ("'Dr Smith is a professor on sabbatical.'",         {"professor","sabbatical","not_teaches"}),
]:
    print(f"  {label:55s} -> {sorted(extension(facts, defaults))}")

print("\nFix: the same rule handles both the implicature and its retraction without")
print("hard-coding exceptions per lexical item.")

## 5. Nested belief reports — "Alice thinks Bob knows that …"

**Task.** Translate sentences with embedded attitude verbs into a representation that distinguishes *what is the case*, *what Alice believes*, and *what Alice believes Bob believes*.

**The bug (classical).** Predicate logic has no operator for "knows" or "believes" — at best you can encode them as opaque relations, losing all the inferences (factivity of *know*, transparency of *believe*, etc.). Famous puzzles (Frege's *Hesperus/Phosphorus*, Kripke's puzzle about belief) cannot even be stated.

**The fix.** Multi-agent epistemic logic. `K_a φ` and `K_a K_b φ` are first-class formulas; we can check each one against a Kripke model.

In [ ]:
task("nested belief reports", "epistemic logic")

@dataclass
class EpiM:
    worlds: Set[str]
    R: Dict[str, Set[Tuple[str,str]]]
    V: Dict[str, Set[str]]
    def holds(self, f, w):
        if f[0]=="atom": return w in self.V.get(f[1], set())
        if f[0]=="not":  return not self.holds(f[1], w)
        if f[0]=="K":
            return all(self.holds(f[2], v) for (u,v) in self.R[f[1]] if u==w)
        raise ValueError(f)

# Two worlds: in w1 the meeting is at 10:00, in w2 it is at 11:00.
# Alice knows the truth (w1). Bob is unsure (sees both). Alice knows that Bob is unsure.
M = EpiM(
    worlds={"w1","w2"},
    R={
        "alice": {("w1","w1"),("w2","w2")},                                       # alice knows
        "bob":   {("w1","w1"),("w1","w2"),("w2","w1"),("w2","w2")},               # bob unsure
    },
    V={"meeting_at_10":{"w1"}},
)

actual = "w1"
atom   = ("atom","meeting_at_10")
queries = [
    ("Alice knows the meeting is at 10",          ("K","alice", atom)),
    ("Bob   knows the meeting is at 10",          ("K","bob",   atom)),
    ("Alice knows that Bob knows the meeting",    ("K","alice",("K","bob", atom))),
    ("Alice knows that Bob does NOT know",        ("K","alice",("not",("K","bob", atom)))),
]
for desc, f in queries:
    print(f"  {desc:45s} -> {M.holds(f, actual)}")

print("\nFix: the system correctly accepts 'Alice knows that Bob does not know',")
print("a nested attitude that classical predicate logic cannot even express.")

## 6. Tense, aspect, and discourse time

**Task.** Verify temporal claims about a narrative — *"He had been waiting until she arrived"*, *"By the time the talk started, the room was already full"*. The natural-language semantics of tense and aspect is exactly the menu of LTL operators.

**The bug (classical).** Plain predicate logic with a `time` argument forces every tense distinction into a manual `t1 < t2` constraint, and aspectual modifiers (*until*, *already*, *eventually*) become ad-hoc bookkeeping.

**The fix.** LTL operators map directly onto English aspect: `F` ≈ *eventually*, `G` ≈ *always*, `U` ≈ *until*, `X` ≈ *next*.

In [ ]:
task("tense and aspect", "LTL")

def ltl(f, trace, i=0):
    n = len(trace)
    if i >= n: return False
    op = f[0]
    if op=="atom": return f[1] in trace[i]
    if op=="not":  return not ltl(f[1], trace, i)
    if op=="and":  return ltl(f[1],trace,i) and ltl(f[2],trace,i)
    if op=="or":   return ltl(f[1],trace,i) or  ltl(f[2],trace,i)
    if op=="imp":  return (not ltl(f[1],trace,i)) or ltl(f[2],trace,i)
    if op=="X":    return i+1<n and ltl(f[1],trace,i+1)
    if op=="F":    return any(ltl(f[1],trace,j) for j in range(i,n))
    if op=="G":    return all(ltl(f[1],trace,j) for j in range(i,n))
    if op=="U":
        for j in range(i,n):
            if ltl(f[2],trace,j): return True
            if not ltl(f[1],trace,j): return False
        return False
    raise ValueError(op)

narrative = [
    {"he_waiting"},
    {"he_waiting"},
    {"he_waiting", "she_arrived"},
    {"they_left"},
    {"they_left", "home"},
]
for i,s in enumerate(narrative): print(f"  t{i}: {s}")

queries = [
    ("He was waiting until she arrived",      ("U", ("atom","he_waiting"), ("atom","she_arrived"))),
    ("Eventually they got home",              ("F", ("atom","home"))),
    ("He was always waiting",                 ("G", ("atom","he_waiting"))),
    ("After she arrived, they left next",     ("F", ("and", ("atom","she_arrived"), ("X",("atom","they_left"))))),
]
for desc, f in queries:
    print(f"  {desc:45s} -> {ltl(f, narrative)}")

print("\nFix: aspectual operators map straight onto LTL, no ad-hoc time-stamp algebra needed.")

## 7. Translation under hedged confidence

**Task.** A translation pipeline produces multiple candidate renderings, each with a different reliability source: a dictionary lookup (high confidence), a phrase-table guess (medium), an LM completion (variable). Hedges in the source — *probably*, *allegedly*, *reportedly* — should propagate to the target.

**The bug (classical).** A flat 'best translation wins' loses both the *hedge* attached to the source sentence and the *reliability* of the candidate. Mixing them gives misleading certainty.

**The fix.** Possibilistic logic. Each candidate carries a necessity weight; source-side hedges multiply the weight; the final translation surfaces with an explicit confidence floor.

In [ ]:
task("translation with hedges", "possibilistic logic")

# Source sentence: 'The minister allegedly resigned.'
# Candidates from different translation channels with their reliability weights.
candidates = [
    ("Le ministre a démissionné.",                  0.95, "dictionary"),
    ("Le ministre aurait démissionné.",             0.80, "phrase_table"),
    ("Le ministre a peut-être démissionné.",        0.55, "LM_completion"),
]
hedge_factor = {
    None:           1.0,
    "reportedly":   0.9,
    "allegedly":    0.7,
    "possibly":     0.5,
}

source_hedge = "allegedly"
h = hedge_factor[source_hedge]
print(f"  Source hedge: {source_hedge!r} (factor {h})")

scored = [(t, min(w, h), src) for t,w,src in candidates]
scored.sort(key=lambda x:-x[1])
for t,w,src in scored:
    print(f"    necessity {w:.2f}  [{src:14s}]  {t}")

print("\nFix: the dictionary translation drops below the phrase-table one *because of the hedge*,")
print("so the system surfaces 'aurait démissionné' (allegedly resigned) as the best match.")

## 8. Toxicity & moderation under annotator disagreement

**Task.** Decide whether a comment is toxic given five human annotators who often disagree. The decision feeds a moderation pipeline that should ask for review when annotators are split, not pick a side.

**The bug (classical).** Majority vote produces a single 0/1 label. A 3–2 split looks identical to a 5–0 consensus, hiding genuine ambiguity from the moderator. Soft labels (e.g. averaging probabilities) hide *uncertainty* in the same way.

**The fix.** Subjective logic. Each annotator's vote becomes an opinion `(b,d,u)`; cumulative fusion gives an aggregate where the residual `u` signals "genuine disagreement, escalate".

In [ ]:
task("moderation under disagreement", "subjective logic")

@dataclass
class Op:
    b: float; d: float; u: float; a: float = 0.5
    def proj(self): return self.b + self.a*self.u

def fuse(o1, o2):
    if o1.u==0 and o2.u==0:
        return Op((o1.b+o2.b)/2, (o1.d+o2.d)/2, 0.0, o1.a)
    den = o1.u + o2.u - o1.u*o2.u
    return Op((o1.b*o2.u + o2.b*o1.u)/den,
              (o1.d*o2.u + o2.d*o1.u)/den,
              (o1.u*o2.u)/den, o1.a)

def from_vote(label, confidence):
    u = 1 - confidence
    if label=="toxic":     return Op(b=confidence, d=0.0, u=u)
    if label=="safe":      return Op(b=0.0, d=confidence, u=u)
    return Op(b=0.0, d=0.0, u=1.0)

examples = [
    ("clear consensus toxic", [("toxic",0.9)]*5),
    ("3-2 split",              [("toxic",0.8)]*3 + [("safe",0.8)]*2),
    ("all unsure",             [("toxic",0.3),("safe",0.3),("toxic",0.3),("safe",0.3),("toxic",0.3)]),
]
for name, votes in examples:
    ops = [from_vote(l,c) for l,c in votes]
    fused = reduce(fuse, ops)
    decision = ("BLOCK" if fused.b > 0.6 else
                "PASS"  if fused.d > 0.6 else
                "REVIEW (uncertainty %.2f)" % fused.u)
    print(f"  {name:24s}  b={fused.b:.2f} d={fused.d:.2f} u={fused.u:.2f}  -> {decision}")

print("\nFix: the 3-2 split routes to human review instead of being silently committed")
print("as a hard label that erases the dissenters.")

## 9. Lexical ambiguity & word-sense fusion

**Task.** Assign a sense to *bank* in *"She walked to the bank to cash a cheque."* Two contextual clues (`walked`, `cheque`) point to different but overlapping sense subsets — a financial institution, a riverside, possibly a building.

**The bug (classical).** Hard sense disambiguation picks one sense and discards the others. When the contextual evidence supports a *family* of senses (e.g. "financial OR business"), forcing a single label loses information that downstream components (translation, retrieval) could have used.

**The fix.** Dempster–Shafer. Each cue distributes belief mass over *subsets* of the sense inventory. Combination narrows the subsets while preserving residual ambiguity.

In [ ]:
task("word-sense fusion", "Dempster–Shafer")

def dempster(m1, m2):
    out, K = {}, 0.0
    for A,pA in m1.items():
        for B,pB in m2.items():
            inter = A & B
            if not inter: K += pA*pB
            else: out[inter] = out.get(inter,0) + pA*pB
    return {s:v/(1-K) for s,v in out.items()}, K

fs = lambda *xs: frozenset(xs)
frame = fs("financial","river","building")

# Cue: 'walked to the bank' — building or river bank, not just abstract financial.
cue_walked = {fs("river","building"):0.6, fs("building"):0.2, frame:0.2}
# Cue: 'cash a cheque' — financial, possibly building.
cue_cheque = {fs("financial"):0.7, fs("financial","building"):0.2, frame:0.1}

fused, K = dempster(cue_walked, cue_cheque)
for s,p in sorted(fused.items(), key=lambda x:-x[1]):
    print(f"    {set(s):35s} {p:.3f}")
print(f"  conflict mass discarded: {K:.3f}")
print("\nFix: the system commits to {financial, building} as the joint sense — i.e.")
print("'a physical bank branch' — not just 'bank.financial' or 'bank.river'.")

## 10. RAG without hallucination — constructive answers

**Task.** A retrieval-augmented QA system should answer *only* claims it can back with a retrieved passage. If no passage entails the answer, the system should say so — not invent a plausible-sounding sentence.

**The bug (classical).** Boolean entailment lets you accept either *p* or *¬p*; with a generative model, neither is grounded in the retrieved evidence, but the LM happily produces one of them. Excluded middle is treated as a *constructive* certainty when in fact no witness exists.

**The fix.** Intuitionistic logic. A claim `p` is *true at state s* only if there is a retrieved passage at `s` that establishes it. `p ∨ ¬p` is rejected when neither has a witness, which is exactly the "I don't know" answer we want.

In [ ]:
task("grounded QA", "intuitionistic logic")

@dataclass
class IntModel:
    states: List[str]
    leq: Set[Tuple[str,str]]
    V: Dict[str, Set[str]]
    def future(self, s): return [t for (u,t) in self.leq if u==s]
    def forces(self, f, s):
        if f[0]=="atom": return s in self.V.get(f[1], set())
        if f[0]=="or":   return self.forces(f[1],s) or self.forces(f[2],s)
        if f[0]=="and":  return self.forces(f[1],s) and self.forces(f[2],s)
        if f[0]=="not":  return all(not self.forces(f[1],t) for t in self.future(s))
        raise ValueError(f)

# States = retrieval rounds. As the retriever finds more passages, more atoms become true.
states = ["q0","q1","q2"]
leq    = {("q0","q0"),("q1","q1"),("q2","q2"),("q0","q1"),("q1","q2"),("q0","q2")}
V      = {
    "einstein_born_1879": {"q1","q2"},     # supported from retrieval round 1
    "einstein_won_nobel": {"q2"},          # only supported in round 2
}
M = IntModel(states, leq, V)

questions = [
    ("Was Einstein born in 1879?",        ("atom","einstein_born_1879")),
    ("Did Einstein win the Nobel?",       ("atom","einstein_won_nobel")),
    ("Did Einstein invent the telephone?",("atom","einstein_invented_phone")),
    ("Either he won the Nobel or didn't", ("or", ("atom","einstein_won_nobel"),
                                                  ("not",("atom","einstein_won_nobel")))),
]
for stage in states:
    print(f"\n  After retrieval stage {stage}:")
    for q, f in questions:
        ans = M.forces(f, stage)
        if ans: tag = "YES (witnessed)"
        else:   tag = "unknown — refuse to answer"
        print(f"    {q:42s} -> {tag}")

print("\nFix: at q0 the system refuses 'either he won or didn't' — there is no witness for")
print("either disjunct, so an intuitionistic agent will not assert excluded middle. As")
print("retrieval progresses (q1, q2), claims become assertible only when grounded.")

## Cross-task summary

| Language task | Logic | What classical NLP missed | What the fix repairs |
|---|---|---|---|
| Vague predicates | fuzzy / Łukasiewicz | sharp threshold, sorites | smooth degree, hedges as functions |
| Contradictory dialogue | LP (paraconsistent) | one contradiction explodes the model | only the disputed atom is flagged |
| Presupposition failure | free logic | phantom referents | explicit `*` triggers a presupposition response |
| Implicature & defaults | default logic | hard-coded exception lists | retraction is automatic from logic alone |
| Nested belief reports | epistemic logic | no `K_a K_b φ` | nested attitudes become first-class formulas |
| Tense & aspect | LTL | manual time-stamp algebra | aspectual operators are native |
| Hedged translation | possibilistic | discards source-hedge & reliability | both propagate as a single weight |
| Annotator disagreement | subjective logic | majority vote hides splits | residual `u` routes to review |
| Word-sense fusion | Dempster–Shafer | one-best sense discards alternatives | belief over sense *subsets* |
| Grounded QA / RAG | intuitionistic | accepts ungrounded answers | only witnessed claims are assertible |

## Take-aways for language pipelines

1. **Vagueness, hedging, and uncertainty are different things.** Fuzzy handles vagueness, possibilistic handles weighted reliability, subjective handles annotator disagreement. Mixing them under one numeric score loses information.
2. **Speakers contradict each other constantly.** A discourse model that treats user input as a flat boolean KB will explode within minutes. Paraconsistent storage is the cheapest fix and costs almost nothing in code.
3. **Definite descriptions need free logic.** Any system that resolves *the X* must handle the case where there is no X. Returning silently `True` or `False` is the source of countless RAG hallucinations.
4. **Defaults are not bugs.** Conversational implicatures are systematic and overridable — exactly what default logic was invented for.
5. **Tense is logic, not metadata.** Once you map *eventually*, *until*, *always* onto LTL operators, narrative comprehension stops needing ad-hoc time arithmetic.
6. **Grounding ≈ intuitionistic witness.** A retrieved passage *is* the constructive witness an intuitionistic logic demands; refusing to assert excluded middle is the same as refusing to hallucinate.

## Open questions specific to language

- Can a fine-tuned LM be trained to *output* possibilistic / subjective logic structures directly, instead of producing flat probabilities that downstream code has to re-interpret?
- How do you compose intuitionistic grounding with default logic — i.e., when is it acceptable for the LM to use a default *without* a retrieved witness?
- Multilingual hedge calibration: the necessity weight of *allegedly* in English ≠ that of *podobno* in Polish or *angeblich* in German. Where should this calibration live?
- Can word-sense fusion via Dempster–Shafer be wired into a vector retriever so that the *sense subset* (not the one-best label) is what gets indexed?